# S05 - Text Classification: Logistic Regression & Naive Bayes
## Solutions

### Exercise 1 (Easy)

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

texts = ["I love this movie", "This movie is terrible", "Great film!", "Waste of time"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("BoW Matrix:\n", X.toarray())

Vocabulary: ['film' 'great' 'is' 'love' 'movie' 'of' 'terrible' 'this' 'time' 'waste']
BoW Matrix:
 [[0 0 0 1 1 0 0 1 0 0]
 [0 0 1 0 1 0 1 1 0 0]
 [1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 1]]


### Exercise 2 (Easy)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(texts)

print("TF-IDF Matrix:\n", X_tfidf.toarray().round(2))

TF-IDF Matrix:
 [[0.   0.   0.   0.67 0.53 0.   0.   0.53 0.   0.  ]
 [0.   0.   0.56 0.   0.44 0.   0.56 0.44 0.   0.  ]
 [0.71 0.71 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.58 0.   0.   0.58 0.58]]


### Exercise 3 (Medium)

In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

texts = ["I love this movie", "Great film", "Excellent acting", "Best movie ever",
         "Terrible movie", "Waste of time", "Awful acting", "Worst film"]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

nb = MultinomialNB()
nb.fit(X, labels)

# Test
test = ["This movie is great", "Terrible waste"]
X_test = vectorizer.transform(test)
print("Predictions:", nb.predict(X_test))

Predictions: [1 0]


### Exercise 4 (Medium)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr = LogisticRegression()
lr.fit(X, labels)

print("Logistic Regression predictions:", lr.predict(X_test))

# Compare with cross-validation
nb_scores = cross_val_score(MultinomialNB(), X, labels, cv=2)
lr_scores = cross_val_score(LogisticRegression(), X, labels, cv=2)

print(f"NB CV Score: {nb_scores.mean():.2f}")
print(f"LR CV Score: {lr_scores.mean():.2f}")

Logistic Regression predictions: [1 0]
NB CV Score: 0.38
LR CV Score: 0.38


### Exercise 5 (Hard)

In [5]:
import math
from collections import defaultdict, Counter

class NaiveBayesClassifier:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_probs = {}
        self.word_probs = defaultdict(dict)
        self.vocab = set()
    
    def fit(self, texts, labels):
        # Count classes
        class_counts = Counter(labels)
        total = len(labels)
        self.class_probs = {c: count/total for c, count in class_counts.items()}
        
        # Build vocabulary and count words per class
        word_counts = defaultdict(Counter)
        for text, label in zip(texts, labels):
            words = text.lower().split()
            self.vocab.update(words)
            word_counts[label].update(words)
        
        # Calculate word probabilities with Laplace smoothing
        V = len(self.vocab)
        for c in class_counts:
            total_words = sum(word_counts[c].values())
            for word in self.vocab:
                count = word_counts[c][word]
                self.word_probs[c][word] = (count + self.alpha) / (total_words + self.alpha * V)
    
    def predict(self, text):
        words = text.lower().split()
        best_class, best_score = None, float('-inf')
        
        for c in self.class_probs:
            score = math.log(self.class_probs[c])
            for word in words:
                if word in self.word_probs[c]:
                    score += math.log(self.word_probs[c][word])
            if score > best_score:
                best_score, best_class = score, c
        return best_class

# Test
nb = NaiveBayesClassifier()
nb.fit(texts, labels)
print(nb.predict("This movie is great"))  # 1
print(nb.predict("Terrible waste"))  # 0

1
0
